In [ ]:
# analysis 3

import neuroimage_analysis as na
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Directories

In [ ]:
dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
outdir = os.path.join(dir, 'results')
schaefer_fc_matrix = np.load(os.path.join(dir, 'data/matrix/mean_adjacency_matrix_1000x1000.npy'))

## Analysis 3: Does sLNM Converge to the Degree Map or PC1?

In real data, the degree map and PC 1 of the reference connectome (C) are highly similar, making it difficult to determine which drives sLNM outputs. To dissociate them, we perform simulations where we explicitly manipulate which principal component (PC) carries the most variance.

**Connectome manipulation via eigenswap**
We downsample from voxel space to 1,000 Schaefer parcels and eigendecompose C, yielding eigenvectors (PCs) and eigenvalues (proportion of variance). By swapping the first and k-th eigenvalues, we reassign maximum variance to any chosen component k, producing a randomized connectome C′.

**Simulation setup**
Each subject's FC map is set to the corresponding row of C′ (lesions constrained to a single Schaefer parcel). Symptoms are modeled from ground truth networks at η² = 0.99. All 1,000 Schaefer parcels are used as ground truth seeds in turn, yielding 1,000 sLNM maps per eigenswap.


In [ ]:
def eigendecom(matrix):
    """Return eigenvalues and eigenvectors of a matrix sorted in descending order."""
    eigenvalues, eigenvectors = np.linalg.eigh(matrix)
    idx = np.argsort(eigenvalues)[::-1]
    return eigenvalues[idx], eigenvectors[:, idx]


def plot_matrix(matrix, title=None):
    """Plot a matrix using a custom diverging colormap."""
    import matplotlib.colors as mcolors

    cmap = mcolors.LinearSegmentedColormap.from_list(
        'custom',
        colors=['cyan', 'blue', 'black', 'red', 'orange', 'yellow'],
        N=256
    )
    plt.imshow(matrix, cmap=cmap, vmin=-0.4, vmax=0.5)
    plt.xticks([])
    plt.yticks([])
    if title is not None:
        plt.title(title)
    plt.show()

In [ ]:
plot_matrix(schaefer_fc_matrix, title = "GSP1000, Schaefer 1000-Parcellations")

# sLNM analyses

In [ ]:
def partial_cor(A, B, C):
    r_AB = pearsonr(A, B)[0]
    r_BC = pearsonr(B, C)[0]
    r_AC = pearsonr(A, C)[0]
    r_AB_C = (r_AB - r_AC * r_BC) / (np.sqrt(1 - r_AC**2) * np.sqrt(1 - r_BC**2))
    return r_AB_C


def bootstrap_dataset(matrix, sample_size=100, eta_sq=0.99, gt_seed=None):
    """
    Generate a bootstrapped FC map and synthetic symptom score dataset for sLNM analysis.

    Randomly selects a ground truth connectome column, correlates bootstrapped
    subjects against it, then scales by effect size and adds noise to produce
    z-scored behavioral scores.

    Parameters
    ----------
    matrix      : (parcels x parcels) connectivity matrix
    sample_size : number of subjects to bootstrap (default: 100)
    eta_sq      : effect size as eta-squared (default: 0.99)
    gt_seed     : column index of ground truth map (random if None)

    Returns
    -------
    dict with 'ground_truth' (parcels,) and 'lnm' (parcels,)
    """
    # Select ground truth map
    if gt_seed is None:
        gt_seed = np.random.choice(matrix.shape[0])
    gt_map = matrix[:, gt_seed]

    # Bootstrap subjects and compute their FC maps
    subject_idx = np.random.choice(matrix.shape[0], sample_size, replace=True)
    subject_map = matrix[:, subject_idx]

    # Generate synthetic behavioral scores based on similarity to ground truth
    r_gt = np.arctanh(np.clip(na.pearson_rows(subject_map.T, gt_map), -1 + 1e-7, 1 - 1e-7)) # prevent NaN

    raw_scores = r_gt * np.sqrt(eta_sq / (1 - eta_sq))
    noisy_scores = raw_scores + np.random.normal(0, 1.0, len(r_gt))
    z_scores = (noisy_scores - np.mean(noisy_scores)) / np.std(noisy_scores)

    lnm_map = na.voxel_outcome_correlation(subject_map.T, z_scores[:, None]).flatten()

    return {
        'ground_truth': gt_map,
        'lnm': lnm_map
    }


def main_analysis(matrix, sample_size=100, effect_size=0.99, seed=42):
    np.random.seed(seed)

    _, PCs = eigendecom(matrix)
    pc1 = PCs[:, 0]
    degree = np.sum(matrix, axis=0)

    # Align sign of pc1 with degree
    if pearsonr(pc1, degree)[0] < 0:
        pc1 = -pc1

    # Generate one sLNM map per parcel
    lnm_maps = np.array([
        bootstrap_dataset(matrix, sample_size=sample_size, eta_sq=effect_size, gt_seed=i)['lnm']
        for i in range(matrix.shape[0])
    ])

    # Full correlations
    pc_lnm_r = []
    degree_lnm_r = []
    # Partial correlations
    partial_pc_given_deg = []
    partial_deg_given_pc = []

    for i in range(lnm_maps.shape[0]):
        pc_lnm_r.append(np.abs(pearsonr(lnm_maps[i], pc1)[0]))
        degree_lnm_r.append(np.abs(pearsonr(lnm_maps[i], degree)[0]))
        partial_pc_given_deg.append(np.abs(partial_cor(lnm_maps[i], pc1, degree)))
        partial_deg_given_pc.append(np.abs(partial_cor(lnm_maps[i], degree, pc1)))

    print(f'sLNM—PC1:              {np.mean(pc_lnm_r):.3f} ± {np.std(pc_lnm_r)/np.sqrt(len(pc_lnm_r)):.3f}')
    print(f'sLNM—Deg:              {np.mean(degree_lnm_r):.3f} ± {np.std(degree_lnm_r)/np.sqrt(len(degree_lnm_r)):.3f}')
    print(f'sLNM—PC1 | Deg:        {np.mean(partial_pc_given_deg):.3f} ± {np.std(partial_pc_given_deg)/np.sqrt(len(partial_pc_given_deg)):.3f}')
    print(f'sLNM—Deg | PC1:        {np.mean(partial_deg_given_pc):.3f} ± {np.std(partial_deg_given_pc)/np.sqrt(len(partial_deg_given_pc)):.3f}')

    return {
        'degree': degree,
        'PC': pc1,
        'pc_lnm_r': pc_lnm_r,
        'degree_lnm_r': degree_lnm_r
    }

In [ ]:
result = main_analysis(schaefer_fc_matrix, effect_size= 0.99)

# Randomize PCs 

In [ ]:
def eigenswap(matrix, n_component):
    """
    Swap the eigenvalue of a target PC with PC1.

    Used to test what sLNM would look like if a different PC
    dominated the connectome's spectral structure.

    Parameters
    ----------
    matrix      : (parcels x parcels) connectivity matrix
    n_component : PC to swap with PC1 (e.g. 2 swaps PC 2 with PC 1)

    Returns
    -------
    Reconstructed symmetric matrix with swapped eigenvalues
    """
    eigenvalues, eigenvectors = eigendecom(matrix)

    # Swap target eigenvalue with PC1 eigenvalue
    target = n_component - 1
    eigenvalues_swapped = eigenvalues.copy()
    eigenvalues_swapped[[0, target]] = eigenvalues_swapped[[target, 0]]

    # Reconstruct and symmetrize matrix
    new_matrix = eigenvectors @ np.diag(eigenvalues_swapped) @ eigenvectors.T
    new_matrix = (new_matrix + new_matrix.T) / 2

    return new_matrix

In [ ]:
# Example: swap PC k with PC 1 and run the main analysis
k = 10

swapped_matrix = eigenswap(schaefer_fc_matrix, n_component=k)
results_k = main_analysis(swapped_matrix)
# Visualize the matrix
plot_matrix(swapped_matrix, title=f'GSP1000, swapped {k}-th PC')
# Visualize degree, PC, and sLNM maps on the left hemisphere
for map_name, map_data in results_k.items():
    if isinstance(map_data, np.ndarray):  # skip scalar values like cluster_size
        na.recon_tmap(map_data, map_name, 'left', 'lateral')

# Numerical Results

In [ ]:
n = 75

deg_pc_corr = []
lnm_pc_corr_mean = []
lnm_pc_corr_sem  = []
lnm_deg_corr_mean  = []
lnm_deg_corr_sem   = []


for k in range(1, n + 1):
    print(f'Analysis k = {k}')
    swapped_matrix = eigenswap(schaefer_fc_matrix, n_component=k)
    results_k = main_analysis(swapped_matrix)

    degree = results_k['degree']
    pc   = results_k['PC']

    deg_pc_corr.append(np.abs(pearsonr(degree, pc)[0]))

    pc_r  = np.array(results_k['pc_lnm_r'])
    deg_r = np.array(results_k['degree_lnm_r'])

    lnm_pc_corr_mean.append(np.mean(pc_r))
    lnm_pc_corr_sem.append(np.std(pc_r) / np.sqrt(len(pc_r)))
    lnm_deg_corr_mean.append(np.mean(deg_r))
    lnm_deg_corr_sem.append(np.std(deg_r) / np.sqrt(len(deg_r)))

  



In [ ]:
# Plot
plt.rcParams['font.family'] = 'Arial'

ks = np.arange(1, n + 1)
mean_pc = np.array(lnm_pc_corr_mean)
sem_pc  = np.array(lnm_pc_corr_sem)
mean_deg  = np.array(lnm_deg_corr_mean)
sem_deg   = np.array(lnm_deg_corr_sem)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ks, deg_pc_corr, label='r(Degree, PC1)', color='gray')
c1 = ax.plot(ks, mean_pc, label='r(sLNM,PC1)')[0].get_color()
ax.fill_between(ks, mean_pc - sem_pc, mean_pc + sem_pc, color=c1, alpha=0.2)
c2 = ax.plot(ks, mean_deg, label='r(sLNM, Degree)')[0].get_color()
ax.fill_between(ks, mean_deg - sem_deg, mean_deg + sem_deg, color=c2, alpha=0.2)
ax.set_xlabel('Shuffled Connectome ($C_k$)', fontsize = 18)
ax.set_ylabel('mean |spatial r|', fontsize = 18)
ax.set_xlim(0.0,1.1)
ax.tick_params(axis = 'y', labelsize = 16)
ax.tick_params(axis = 'x', labelsize = 16)
ax.legend(fontsize = 14)
ax.set_xlim(1, n)
fig.tight_layout()
plt.show()